## 3. cleaning dataset

In [1]:
import polars as pl


df = pl.read_csv("eth_tx_last4days_2.csv")


df.head()

block_number,hash,from_address,to_address,value,gas,gas_price,receipt_gas_used,block_timestamp
i64,str,str,str,f64,i64,i64,i64,str
23719749,"""0x62129d0a1c1bdd8ef11bb808066f…","""0xef317e433b0836f294866d43f67d…","""0xb58850a28cbef446071fccbe71bd…",8.4973e17,90000,17122619816,21000,"""2025-11-03 15:38:35+00:00"""
23719749,"""0xa235f7c937bfdd82bd144c3bd914…","""0xb0d035a95f2057e88367befc5f93…","""0x644e75be0997898e5e7286b95612…",2.5319e15,21000,16156013539,21000,"""2025-11-03 15:38:35+00:00"""
23719749,"""0x82f9a03526bbf801cf0ec26d6483…","""0x7eab937302c35669ac5217bcc81d…","""0xc02aaa39b223fe8d0a0e5c4f27ea…",0.0,77665,16788903463,46806,"""2025-11-03 15:38:35+00:00"""
23719749,"""0xa240b11ebd2f8dd29416757c21fe…","""0x303fc5322ff01b9dbba8b57ca9e6…","""0xdac17f958d2ee523a22062069945…",0.0,100000,24394265433,46109,"""2025-11-03 15:38:35+00:00"""
23719749,"""0xea2ec158d3c1d4c6a2dbeb259f2b…","""0x974caa59e49682cda0ad2bbe8298…","""0x60853b40234f5bd5598a08ece98e…",2.4624e16,105000,16416580066,21000,"""2025-11-03 15:38:35+00:00"""


In [2]:
events = (
    df
    .select([
        pl.col("hash"),
        pl.col("from_address").str.to_lowercase().alias("source"),
        pl.col("to_address").str.to_lowercase().alias("target"),
        pl.col("value").alias("value_wei"),
        pl.col("block_timestamp")
    ])
    .with_columns([
        pl.col("target").fill_null("__contract_creation__"),
        (
            pl.col("block_timestamp")
            .str.to_datetime(format="%Y-%m-%d %H:%M:%S%z")
            .dt.timestamp("ms") // 1000
        ).alias("timestamp"),
        (
            pl.col("value_wei").cast(pl.Float64, strict=False) / 1_000_000_000_000_000_000
        ).alias("value_eth")
    ])
    .select([
        "hash",
        "source",
        "target",
        "timestamp",
        "block_timestamp",
        "value_wei",
        "value_eth"
    ])
    .sort("timestamp")
)

events.head()

hash,source,target,timestamp,block_timestamp,value_wei,value_eth
str,str,str,i64,str,f64,f64
"""0x5fbbfb290acb067a7563b2405af2…","""0x3ee2b94eafb817c21db88b3389b9…","""0x0c1f3412a44ff99e40bf14e06e5e…",1761838727,"""2025-10-30 15:38:47+00:00""",0.0,0.0
"""0x9ed4f854052ed1372f3c5e74f28f…","""0x0530145186c0c73d6a194080d0ed…","""0xbbbfd134e9b44bfb5123898ba36b…",1761838727,"""2025-10-30 15:38:47+00:00""",0.0,0.0
"""0x4538acbdd2d52f9484372518aeb4…","""0x7017003dfb632dcf9189525f9a35…","""0xdac17f958d2ee523a22062069945…",1761838727,"""2025-10-30 15:38:47+00:00""",0.0,0.0
"""0xfe6c7edcc5a30a2ae0c0dd350daa…","""0x315d2ee4fccda0def532ef4108ff…","""0xc46fcd651bd6ac11255886feabdc…",1761838727,"""2025-10-30 15:38:47+00:00""",0.0,0.0
"""0xad6cc26e8b7d9122af37257ecb71…","""0xa49999992755dcc3a42cb10348b1…","""0xa06ecaabe5c6d9a86cb5a2a618e1…",1761838727,"""2025-10-30 15:38:47+00:00""",7.8000e15,0.0078


In [3]:
print(events.shape)
print(events.schema)

events.select([
    pl.col("timestamp").null_count().alias("missing_timestamp"),
    pl.col("source").null_count().alias("missing_source"),
    pl.col("target").null_count().alias("missing_target"),
    pl.col("value_wei").null_count().alias("missing_value_wei"),
    pl.col("value_eth").null_count().alias("missing_value_eth")
])

(5992184, 7)
Schema({'hash': String, 'source': String, 'target': String, 'timestamp': Int64, 'block_timestamp': String, 'value_wei': Float64, 'value_eth': Float64})


missing_timestamp,missing_source,missing_target,missing_value_wei,missing_value_eth
u32,u32,u32,u32,u32
0,0,0,0,0


In [4]:
events.select([
    pl.col("timestamp").min().alias("min_timestamp"),
    pl.col("timestamp").max().alias("max_timestamp"),
    pl.col("block_timestamp").first().alias("first_block_timestamp"),
    pl.col("block_timestamp").last().alias("last_block_timestamp"),
    pl.col("value_eth").min().alias("min_value_eth"),
    pl.col("value_eth").max().alias("max_value_eth"),
    pl.col("value_eth").mean().alias("mean_value_eth")
])

min_timestamp,max_timestamp,first_block_timestamp,last_block_timestamp,min_value_eth,max_value_eth,mean_value_eth
i64,i64,str,str,f64,f64,f64
1761838727,1762184315,"""2025-10-30 15:38:47+00:00""","""2025-11-03 15:38:35+00:00""",0.0,84642.407544,1.055497


In [5]:
# --- STEP 0: PREPARE ACTIVE NODES TABLE ---
# We create a base table with all unique addresses found in 'events'
print("Collecting active addresses from transactions...")

# Extract unique addresses from both source and target columns
# Then combine and keep unique ones only
nodes = (
    pl.concat([
        events.select(pl.col("source").alias("address")),
        events.select(pl.col("target").alias("address"))
    ])
    .unique()
)

print(f"Base 'nodes' table created. Total active nodes: {nodes.height}")

Base 'nodes' table created. Total active nodes: 1746324


## 5. repeated_same_direction motif

In [6]:
import polars as pl

# --- CONFIGURATION ---
# Time window delta (1 hour as per your notebook)
DELTA_SECONDS = 3600 

# --- STEP 1: REPEATED SAME DIRECTION MOTIF ---
# Logic: Counting cases where A sends to B multiple times within delta
print("Calculating: Repeated Same Direction...")

# 1. Sort by pair and time to align transactions in sequence
# 2. Use 'shift' to look at the previous transaction within the same [source, target] pair
repeated_same_events = (
    events
    .sort(["source", "target", "timestamp"])
    .with_columns([
        pl.col("timestamp").shift(1).over(["source", "target"]).alias("prev_ts"),
        pl.col("value_eth").shift(1).over(["source", "target"]).alias("prev_val")
    ])
    # Identify if current transaction is within 1h of the previous one
    .with_columns(
        (pl.col("timestamp") - pl.col("prev_ts") <= DELTA_SECONDS).alias("is_motif")
    )
    .filter(pl.col("is_motif") == True)
)

# Aggregate results to the node level
# We calculate features for both SENDER (source) and RECEIVER (target) roles
repeated_sender_features = (
    repeated_same_events
    .group_by("source")
    .agg([
        pl.len().alias("repeat_same_direction_as_sender_count_1h"),
        pl.col("value_eth").sum().alias("repeat_same_direction_as_sender_value_eth_1h")
    ])
    .rename({"source": "address"})
)

repeated_receiver_features = (
    repeated_same_events
    .group_by("target")
    .agg([
        pl.len().alias("repeat_same_direction_as_receiver_count_1h"),
        pl.col("value_eth").sum().alias("repeat_same_direction_as_receiver_value_eth_1h")
    ])
    .rename({"target": "address"})
)

# Join results back to our main 'nodes' table
nodes = (
    nodes
    .join(repeated_sender_features, on="address", how="left")
    .join(repeated_receiver_features, on="address", how="left")
    .fill_null(0)
)

print("Step 1 complete: Repeated Same Direction features added to 'nodes'.")

Calculating: Repeated Same Direction...
Step 1 complete: Repeated Same Direction features added to 'nodes'.


In [7]:
nodes.shape

(1746324, 5)

## 6. reciprocity motif

In [8]:
# --- STEP 2: RECIPROCITY (A -> B and B -> A within 1h) ---
# We avoid Join to prevent Kernel Crash. Instead, we use sorting and shifting.
print("Calculating: Reciprocity (Memory Safe)...")

# 1. Create a canonical key for each pair of addresses
# So that A->B and B->A share the same pair_id
reciprocity_data = (
    events
    .with_columns([
        pl.when(pl.col("source") < pl.col("target"))
        .then(pl.col("source"))
        .otherwise(pl.col("target"))
        .alias("p1"),
        pl.when(pl.col("source") < pl.col("target"))
        .then(pl.col("target"))
        .otherwise(pl.col("source"))
        .alias("p2")
    ])
    # 2. Sort by the pair and time
    .sort(["p1", "p2", "timestamp"])
    # 3. Look at the previous transaction in this pair
    .with_columns([
        pl.col("source").shift(1).over(["p1", "p2"]).alias("prev_source"),
        pl.col("timestamp").shift(1).over(["p1", "p2"]).alias("prev_ts"),
        pl.col("value_eth").shift(1).over(["p1", "p2"]).alias("prev_val")
    ])
    # 4. Identify reciprocity: 
    # Current source must be different from previous source (meaning B -> A followed A -> B)
    .with_columns(
        ((pl.col("source") != pl.col("prev_source")) & 
         (pl.col("timestamp") - pl.col("prev_ts") <= DELTA_SECONDS))
        .alias("is_reciprocity")
    )
    .filter(pl.col("is_reciprocity") == True)
)

# 5. Aggregate stats for the address that completed the reciprocity (sender of the second leg)
recip_features = (
    reciprocity_data
    .group_by("source")
    .agg([
        pl.len().alias("motif_reciprocity_cnt"),
        pl.col("value_eth").sum().alias("motif_reciprocity_val_sum")
    ])
    .rename({"source": "address"})
)

# 6. Update our 'nodes' table
nodes = (
    nodes
    .join(recip_features, on="address", how="left")
    .fill_null(0)
)

print(f"Step 2 complete. Current columns: {nodes.columns}")

Calculating: Reciprocity (Memory Safe)...
Step 2 complete. Current columns: ['address', 'repeat_same_direction_as_sender_count_1h', 'repeat_same_direction_as_sender_value_eth_1h', 'repeat_same_direction_as_receiver_count_1h', 'repeat_same_direction_as_receiver_value_eth_1h', 'motif_reciprocity_cnt', 'motif_reciprocity_val_sum']


## 7. chain motif

In [9]:
from collections import defaultdict
from bisect import bisect_left, bisect_right

# --- Вспомогательные функции из твоего ноутбука ---
# (Я немного упростил их, чтобы они возвращали только нужные нам колонки)

def count_chain_middle_safe(group, delta=3600):
    # group: [address, direction, timestamp, value_eth]
    address = group["address"][0]
    incoming = group.filter(pl.col("direction") == "incoming").sort("timestamp")
    outgoing = group.filter(pl.col("direction") == "outgoing").sort("timestamp")
    
    if incoming.height == 0 or outgoing.height == 0:
        return pl.DataFrame({"address": [address], "motif_chain_cnt": [0]})

    in_times = incoming["timestamp"].to_list()
    out_times = outgoing["timestamp"].to_list()
    
    chain_count = 0
    for t_in in in_times:
        # Ищем сколько исходящих было в течение часа после входящего
        left = bisect_right(out_times, t_in)
        right = bisect_right(out_times, t_in + delta)
        chain_count += (right - left)
        
    return pl.DataFrame({"address": [address], "motif_chain_cnt": [chain_count]})

# --- РАСЧЕТ ---

print("Calculating: Chain Motif (Memory Safe)...")

# 1. Готовим данные для цепочек
chain_input = pl.concat([
    events.select([
        pl.col("target").alias("address"),
        pl.lit("incoming").alias("direction"),
        pl.col("timestamp"),
        pl.col("value_eth")
    ]),
    events.select([
        pl.col("source").alias("address"),
        pl.lit("outgoing").alias("direction"),
        pl.col("timestamp"),
        pl.col("value_eth")
    ])
])

# 2. Считаем Chain через map_groups
chain_features = (
    chain_input
    .group_by("address")
    .map_groups(lambda g: count_chain_middle_safe(g, delta=3600))
)

# 3. Считаем Fan-In и Fan-Out (простой способ без rolling)
# В Ethereum "Веер" — это когда у адреса много уникальных контрагентов за короткое время.
# Мы посчитаем общее кол-во уникальных связей, как в упрощенном графовом анализе.

print("Calculating: Fan-In and Fan-Out...")
fan_in_features = (
    events.group_by("target")
    .agg([
        pl.col("source").n_unique().alias("motif_fan_in_unique_sources"),
        pl.len().alias("motif_fan_in_cnt")
    ])
    .rename({"target": "address"})
)

fan_out_features = (
    events.group_by("source")
    .agg([
        pl.col("target").n_unique().alias("motif_fan_out_unique_targets"),
        pl.len().alias("motif_fan_out_cnt")
    ])
    .rename({"source": "address"})
)

# 4. Собираем всё в таблицу nodes
nodes = (
    nodes
    .join(chain_features, on="address", how="left")
    .join(fan_in_features, on="address", how="left")
    .join(fan_out_features, on="address", how="left")
    .fill_null(0)
)

print(f"Done! Current nodes shape: {nodes.shape}")
print(f"Columns: {nodes.columns}")

Calculating: Chain Motif (Memory Safe)...
Calculating: Fan-In and Fan-Out...
Done! Current nodes shape: (1746324, 12)
Columns: ['address', 'repeat_same_direction_as_sender_count_1h', 'repeat_same_direction_as_sender_value_eth_1h', 'repeat_same_direction_as_receiver_count_1h', 'repeat_same_direction_as_receiver_value_eth_1h', 'motif_reciprocity_cnt', 'motif_reciprocity_val_sum', 'motif_chain_cnt', 'motif_fan_in_unique_sources', 'motif_fan_in_cnt', 'motif_fan_out_unique_targets', 'motif_fan_out_cnt']


## 10. cycle motif

In [11]:
import polars as pl
from collections import defaultdict
from bisect import bisect_left, bisect_right

def empty_cycle_contribs():
    return pl.DataFrame({
        "address": pl.Series([], dtype=pl.Utf8),
        "cycle_as_start_count_1h": pl.Series([], dtype=pl.Int64),
        "cycle_as_start_value_eth_1h": pl.Series([], dtype=pl.Float64),
        "cycle_as_middle_count_1h": pl.Series([], dtype=pl.Int64),
        "cycle_as_middle_value_eth_1h": pl.Series([], dtype=pl.Float64),
        "cycle_as_end_count_1h": pl.Series([], dtype=pl.Int64),
        "cycle_as_end_value_eth_1h": pl.Series([], dtype=pl.Float64)
    })

def build_cycle_indexes(tx):
    incoming_by_target = defaultdict(lambda: {"times": [], "sources": [], "values": []})
    closing_by_pair = defaultdict(lambda: {"times": [], "values": []})

    tx_sorted = tx.sort("timestamp")

    for source, target, timestamp, value_eth in tx_sorted.iter_rows():
        incoming_by_target[target]["times"].append(timestamp)
        incoming_by_target[target]["sources"].append(source)
        incoming_by_target[target]["values"].append(value_eth)

        closing_by_pair[(source, target)]["times"].append(timestamp)
        closing_by_pair[(source, target)]["values"].append(value_eth)

    closing_prefix_by_pair = {}
    for pair, data in closing_by_pair.items():
        prefix = [0.0]
        for value in data["values"]:
            prefix.append(prefix[-1] + value)
        closing_prefix_by_pair[pair] = prefix

    return incoming_by_target, closing_by_pair, closing_prefix_by_pair

def compute_cycle_contribs(tx, delta=3600):
    # Explicitly keep only the 4 required columns to prevent unpacking errors
    tx = tx.select(["source", "target", "timestamp", "value_eth"])

    incoming_by_target, closing_by_pair, closing_prefix_by_pair = build_cycle_indexes(tx)

    contrib = defaultdict(lambda: [0, 0.0, 0, 0.0, 0, 0.0])
    tx_second_edges = tx.sort("timestamp")

    for row in tx_second_edges.iter_rows(named=True):
        b = row["source"]
        c = row["target"]
        t2 = row["timestamp"]
        v2 = row["value_eth"]

        incoming_data = incoming_by_target.get(b)
        if incoming_data is None:
            continue

        in_times = incoming_data["times"]
        in_sources = incoming_data["sources"]
        in_values = incoming_data["values"]

        left_in = bisect_left(in_times, t2 - delta)
        right_in = bisect_left(in_times, t2)

        if left_in == right_in:
            continue

        for i in range(left_in, right_in):
            a = in_sources[i]
            t1 = in_times[i]
            v1 = in_values[i]

            if a == b or b == c or a == c:
                continue

            closing_pair = (c, a)
            closing_data = closing_by_pair.get(closing_pair)
            if closing_data is None:
                continue

            close_times = closing_data["times"]
            close_prefix = closing_prefix_by_pair[closing_pair]

            left_close = bisect_right(close_times, t2)
            right_close = bisect_right(close_times, t1 + delta)

            k = right_close - left_close
            if k <= 0:
                continue

            closing_value_sum = close_prefix[right_close] - close_prefix[left_close]
            motif_value_sum = k * (v1 + v2) + closing_value_sum

            contrib[a][0] += k
            contrib[a][1] += motif_value_sum

            contrib[b][2] += k
            contrib[b][3] += motif_value_sum

            contrib[c][4] += k
            contrib[c][5] += motif_value_sum

    if len(contrib) == 0:
        return empty_cycle_contribs()

    rows = []
    for address, values in contrib.items():
        rows.append((
            address,
            values[0],
            values[1],
            values[2],
            values[3],
            values[4],
            values[5]
        ))

    return pl.DataFrame(
        rows,
        schema=[
            "address",
            "cycle_as_start_count_1h",
            "cycle_as_start_value_eth_1h",
            "cycle_as_middle_count_1h",
            "cycle_as_middle_value_eth_1h",
            "cycle_as_end_count_1h",
            "cycle_as_end_value_eth_1h"
        ],
        orient="row"
    )

print("Calculating: Cycle Motif (Global Index approach)...")
cycle_features = compute_cycle_contribs(events, delta=3600)

nodes = (
    nodes
    .join(cycle_features, on="address", how="left")
    .fill_null(0)
)

print(f"Step 6 complete. Final nodes shape: {nodes.shape}")
print(f"Columns: {nodes.columns}")

Calculating: Cycle Motif (Global Index approach)...
Step 6 complete. Final nodes shape: (1746324, 18)
Columns: ['address', 'repeat_same_direction_as_sender_count_1h', 'repeat_same_direction_as_sender_value_eth_1h', 'repeat_same_direction_as_receiver_count_1h', 'repeat_same_direction_as_receiver_value_eth_1h', 'motif_reciprocity_cnt', 'motif_reciprocity_val_sum', 'motif_chain_cnt', 'motif_fan_in_unique_sources', 'motif_fan_in_cnt', 'motif_fan_out_unique_targets', 'motif_fan_out_cnt', 'cycle_as_start_count_1h', 'cycle_as_start_value_eth_1h', 'cycle_as_middle_count_1h', 'cycle_as_middle_value_eth_1h', 'cycle_as_end_count_1h', 'cycle_as_end_value_eth_1h']


In [12]:
output_file_path = "nodes_with_temporal_motifs_1h.csv"

print(f"Saving final features to: {output_file_path}...")
nodes.write_csv(output_file_path)

print("File successfully saved!")
print(f"Final dataset shape: {nodes.shape}")
print(f"Available columns: {nodes.columns}")

Saving final features to: nodes_with_temporal_motifs_1h.csv...
File successfully saved!
Final dataset shape: (1746324, 18)
Available columns: ['address', 'repeat_same_direction_as_sender_count_1h', 'repeat_same_direction_as_sender_value_eth_1h', 'repeat_same_direction_as_receiver_count_1h', 'repeat_same_direction_as_receiver_value_eth_1h', 'motif_reciprocity_cnt', 'motif_reciprocity_val_sum', 'motif_chain_cnt', 'motif_fan_in_unique_sources', 'motif_fan_in_cnt', 'motif_fan_out_unique_targets', 'motif_fan_out_cnt', 'cycle_as_start_count_1h', 'cycle_as_start_value_eth_1h', 'cycle_as_middle_count_1h', 'cycle_as_middle_value_eth_1h', 'cycle_as_end_count_1h', 'cycle_as_end_value_eth_1h']
